# 2. Sequential Preprocessing and Tokenization

This notebook builds a **custom domain-specific tokenizer** for financial transactions and generates tokenized corpora for foundation model pretraining.

Instead of treating each transaction as an independent row, we serialize a customer's transaction history into **a sequence** of domain-specific tokens so a language model can learn sequential patterns (spending velocity, geographic shifts, recurring merchants).

* **Custom tokenization** -- why domain-specific tokens outperform general-purpose BPE tokenizers for financial data.
* **Corpus generation** -- convert the temporal splits from notebook 01 into text sequences for foundation model pretraining.

In [1]:
import pandas as pd
import numpy as np
import time

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch.cuda")

In [2]:
!pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com
!pip install torch
!pip install "pyarrow==23.0.0"

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 36.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


In [2]:
import sys
# Update this path to the directory containing model.py
sys.path.append('/content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/')

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ls /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/data/

Mounted at /content/drive
embeddings  test_eval.parquet  train.parquet	 val.parquet
outputs     test.parquet       val_eval.parquet


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/data/Foundation-Transaction/card_transaction.v1.csv")
df.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [4]:
TEMPORAL_SPLIT_DIR = "/content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/data"

TRAIN_DATA = f"{TEMPORAL_SPLIT_DIR}/train.parquet"
VAL_DATA   = f"{TEMPORAL_SPLIT_DIR}/val.parquet"
TEST_DATA  = f"{TEMPORAL_SPLIT_DIR}/test.parquet"

# for p in [TRAIN_DATA, VAL_DATA, TEST_DATA]:
#     assert p.exists(), f"Missing temporal split: {p}  (run notebook 01 first)"

CORPUS_DIR = "/content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus"
CORPUS_PATH      = f"{CORPUS_DIR}/train_corpus.txt"
VAL_CORPUS_PATH  = f"{CORPUS_DIR}/val_corpus.txt"
TEST_CORPUS_PATH = f"{CORPUS_DIR}/test_corpus.txt"

# from src.tokenizer import FinancialTokenizerPipeline, FinancialTabularTokenizer

# print(f"Project Root      : {PROJECT_ROOT}")
print(f"Temporal Splits   : {TEMPORAL_SPLIT_DIR}")
print(f"  Train           : {TRAIN_DATA}")
print(f"  Val             : {VAL_DATA}")
print(f"  Test            : {TEST_DATA}")
print(f"Output Corpora    : {CORPUS_DIR}")
print(f"  Train Corpus    : {CORPUS_PATH}")
print(f"  Val Corpus      : {VAL_CORPUS_PATH}")
print(f"  Test Corpus     : {TEST_CORPUS_PATH}")

Temporal Splits   : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/data
  Train           : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/data/train.parquet
  Val             : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/data/val.parquet
  Test            : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/data/test.parquet
Output Corpora    : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus
  Train Corpus    : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/train_corpus.txt
  Val Corpus      : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/val_corpus.txt
  Test Corpus     : /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/test_corpus.txt


## Tokenization Deep Dive

Standard LLM tokenizers are optimized for natural language. For tabular financial data, they often fail to capture semantic meaning efficiently:

* **Amounts**: $\$123.45$ might be split into ['$', '123', '.', '45'], losing magnitude information.
* **Merchants**: Walmart #8832 might be split into subwords, losing the entity identity.
* **Timestamps**: Dates are split into arbitrary numbers.

Our Financial Tokenizer uses domain-specific logic:

* **Amount Binning**: Maps amounts to log-scale **bins** (e.g., $ \$42.75$ = AMT_1; AMT_3 = $100-500).
* **Merchant Hashing**: Hashes merchant names to a fixed vocabulary (e.g., MERCH_1234).
* **Temporal Encoding**: Encodes time as HOUR_XX, DOW_X, MONTH_XX.
* **Location Decomposition**: Splits location into ZIP3_xxx (region) + STATE_xx.
* **Privacy**:

### 1.1 Initialize Tokenizer

We initialize our custom tokenizer and load a standard **GPT-2 BPE** tokenizer for comparison.

In [5]:
import torch
import cudf
from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"GPT-2 Vocab Size    : {gpt2_tokenizer.vocab_size}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT-2 Vocab Size    : 50257


#### Financial Tokenizer

In [6]:
from src.tokenizer import (
    FinancialTokenizerPipeline,
    FinancialTabularTokenizer,
)

pipeline = FinancialTokenizerPipeline(merchant_hash_size=2000)
print(f"  Pipeline steps    : {len(pipeline.tokenizer_order)}")
print(f"  Step names        : {pipeline.tokenizer_order}")

fin_tokenizer = FinancialTabularTokenizer(merchant_hash_size=2000)
print(f"Financial Vocab Size: {fin_tokenizer.get_vocab_size()}")

  Pipeline steps    : 12
  Step names        : ['amt_val', 'merch_hash', 'mcc_int', 'mcc_str', 'hour', 'dow', 'month', 'card', 'chip_upper', 'zip3', 'state_clean', 'cust']
Financial Vocab Size: 6251


### 1.2 Sample Transaction

Tokenize a single transaction using the TabFormer schema to see the pipeline output.

In [7]:
# Build a single-row cuDF DataFrame matching the TabFormer schema
sample_df = cudf.DataFrame({
    "Amount":         ["$42.75"],
    "Merchant Name":  ["3527213246127876953"],
    "MCC":            [5411],
    "Year":           [2025],
    "Month":          [1],
    "Day":            [15],
    "Time":           ["09:32"],
    "Card":           [0],
    "Use Chip":       ["Chip Transaction"],
    "Zip":            ["95113.0"],
    "Merchant State": ["CA"],
    "User":           [1001],
})

# Run the full pipeline: preprocess → fit → transform
processed = FinancialTokenizerPipeline.preprocess(sample_df)
pipeline.fit(processed)
token_df = pipeline.transform(processed)

# Show per-field tokens
print("Pipeline token output (one column per field):")
print(token_df.to_pandas().head())

print('------------------------')

# Assemble into the text format used by the decoder training corpus
token_cols = list(token_df.columns)
txn_str = " ".join(token_df[c].to_pandas().iloc[0] for c in token_cols)
financial_text = f"<bos> {txn_str} <eos>"
print(f"\nFull token sequence:\n{financial_text}")

Pipeline token output (one column per field):
  amt_val merch_hash     mcc_int   mcc_str     hour    dow     month    card  \
0   AMT_1  MERCH_898  CAT_RETAIL  MCC_5411  HOUR_09  DOW_2  MONTH_01  CARD_0   

  chip_upper      zip3 state_clean       cust  
0  CHIP_CHIP  ZIP3_951    STATE_CA  CUST_1001  
------------------------

Full token sequence:
<bos> AMT_1 MERCH_898 CAT_RETAIL MCC_5411 HOUR_09 DOW_2 MONTH_01 CARD_0 CHIP_CHIP ZIP3_951 STATE_CA CUST_1001 <eos>


### 1.3 Tokenizer Comparison

How would a general-purpose BPE tokenizer handle the same transaction? We compare **our Financial Tokenizer** against **GPT-2** applied to a raw serialization of the tabular fields. The token count directly determines how many transactions fit into the model's fixed context window.



In [ ]:
sample_df.head()

,amount,merchant_name,mcc,year,month,day,time,card,use_chip,zip,...,merch_hash,mcc_int,mcc_str,hour,dow,chip_upper,zip3,state_clean,cust,time_full
0,$42.75,3527213246127876953,5411,2025,1,15,09:32,0,Chip Transaction,95113.0,...,1297092898,5411,5411,9,2,CHIP TRANSACTION,951,CA,1001,2025-01-15 09:32:00


In [8]:
fin_tokens = financial_text.split()

raw_tabular_text = (
    "$42.75, 3527213246127876953, 5411, "
    "2025-01-15, 09:32, 0, Chip Transaction, "
    "95113, CA, 1001"
)

print("=" * 70)
print("TOKENIZER COMPARISON: Same Transaction, Two Approaches")
print("=" * 70)

print(f"\n--- Financial Tokenizer (domain-specific pipeline) ---")
print(f"Input : raw tabular fields → binning + hashing + encoding")
print(f"Count : {len(fin_tokens)} tokens")
print(f"Tokens: {fin_tokens}\n")

print()
print("*" * 50)
print()

gpt2_tokens = gpt2_tokenizer.tokenize(raw_tabular_text)
print(f"--- GPT-2 BPE Tokenizer (on raw tabular text) ---")
print(f"Input : {raw_tabular_text}")
print(f"Count : {len(gpt2_tokens)} tokens")
print(f"Tokens: {gpt2_tokens}")


TOKENIZER COMPARISON: Same Transaction, Two Approaches

--- Financial Tokenizer (domain-specific pipeline) ---
Input : raw tabular fields → binning + hashing + encoding
Count : 14 tokens
Tokens: ['<bos>', 'AMT_1', 'MERCH_898', 'CAT_RETAIL', 'MCC_5411', 'HOUR_09', 'DOW_2', 'MONTH_01', 'CARD_0', 'CHIP_CHIP', 'ZIP3_951', 'STATE_CA', 'CUST_1001', '<eos>']


**************************************************

--- GPT-2 BPE Tokenizer (on raw tabular text) ---
Input : $42.75, 3527213246127876953, 5411, 2025-01-15, 09:32, 0, Chip Transaction, 95113, CA, 1001
Count : 39 tokens
Tokens: ['$', '42', '.', '75', ',', 'Ġ35', '27', '213', '246', '12', '787', '69', '53', ',', 'Ġ54', '11', ',', 'Ġ2025', '-', '01', '-', '15', ',', 'Ġ09', ':', '32', ',', 'Ġ0', ',', 'ĠChip', 'ĠTransaction', ',', 'Ġ95', '113', ',', 'ĠCA', ',', 'Ġ100', '1']


### 1.4 Quantitative Summary
The token count difference has a dramatic impact on what the model can learn.

In [9]:
fin_count = len(fin_tokens)
gpt2_count = len(gpt2_tokens)

CONTEXT_WINDOW = 4096
TOKENS_PER_TXN_SEP = 1

fin_per_txn = fin_count - 2  # subtract <bos>/<eos> (per-sequence overhead, not per-txn)
fin_txns_per_seq = CONTEXT_WINDOW // (fin_per_txn + TOKENS_PER_TXN_SEP)
gpt2_txns_per_seq = CONTEXT_WINDOW // (gpt2_count + TOKENS_PER_TXN_SEP)

print("=" * 70)
print("TOKENIZER COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Tokenizer':<20} {'Tokens/Txn':>12} {'Compression':>14} {'Txns in 4096':>15}")
print("-" * 70)
print(f"{'Financial':<20} {fin_per_txn:>12} {'1x (baseline)':>14} {f'~{fin_txns_per_seq}':>15}")
print(f"{'GPT-2 (BPE)':<20} {gpt2_count:>12} {f'{gpt2_count/fin_per_txn:.1f}x more':>14} {f'~{gpt2_txns_per_seq}':>15}")
print("-" * 70)
print(f"\nWith a 4096-token context window:")
print(f"  Financial tokenizer: ~{fin_txns_per_seq} transactions of temporal context")
print(f"  GPT-2 BPE tokenizer: ~{gpt2_txns_per_seq} transactions")


TOKENIZER COMPARISON SUMMARY
Tokenizer              Tokens/Txn    Compression    Txns in 4096
----------------------------------------------------------------------
Financial                      12  1x (baseline)            ~315
GPT-2 (BPE)                    39      3.2x more            ~102
----------------------------------------------------------------------

With a 4096-token context window:
  Financial tokenizer: ~315 transactions of temporal context
  GPT-2 BPE tokenizer: ~102 transactions


### 1.5 Analysis: Why This Matters

The Financial Tokenizer maps each field to one semantically meaningful token, while BPE tokenizers fragment raw values into subwords:

| Concept	| Financial Tokenizer	| GPT-2 BPE on raw text	| Problem with BPE |
| --- | --- | --- | --- |
| Amount |	AMT_1 (1 token) |	$\$42$, ., 75 (3+ tokens)|	Magnitude is lost -- $ \$42$ and $\$4,200$ produce similar fragments|
|Merchant ID	| MERCH_667 (1 token)	| 352, 72, 13, ... (10+ tokens)	| A 19-digit number explodes into meaningless digit subwords
| Time of day	| HOUR_09 (1 token) |	09, :, 32 (3 tokens)|	Hour and minute are separated; temporal grouping must be learned |
| Location |	ZIP3_951 + STATE_CA (2 tokens) |	95, 113, ,, CA (4+ tokens)	| Full ZIP is exposed; no privacy-preserving truncation

With a 4096-token context window, standard BPE tokenizers fit ~80-130 transactions per sequence. Our financial tokenizer fits ~315 transactions (12 tokens each), giving the model access to months of spending history. The tokenization also provides a privacy layer: merchant names are hashed, amounts are binned, and ZIP codes are truncated to 3 digits.

### 1.6 Edge Cases

In [10]:
def quick_tokenize(overrides: dict) -> str:
    """Helper: tokenize a single transaction with field overrides."""
    row = {
        "Amount": ["$42.75"],
        "Merchant Name": ["3527213246127876953"],
        "MCC": [5411],
        "Year": [2025],
        "Month": [1],
        "Day": [15],
        "Time": ["09:32"],
        "Card": [0],
        "Use Chip": ["Chip Transaction"],
        "Zip": ["95113.0"],
        "Merchant State": ["CA"],
        "User": [1001],
    }
    row.update({k: [v] for k, v in overrides.items()})
    gdf = cudf.DataFrame(row)
    gdf = FinancialTokenizerPipeline.preprocess(gdf)
    tdf = pipeline.transform(gdf)
    return " ".join(tdf[c].to_pandas().iloc[0] for c in tdf.columns)

print("Large amount ($8,500):")
tokens = quick_tokenize({"Amount": "$8500.00"})
print(tokens)
print(f"  {tokens.split()[0]}  (AMT_6 = $5,000+)")

print("*" * 50)

print("\nSmall amount ($0.50):")
tokens = quick_tokenize({"Amount": "$0.50"})
print(tokens)
print(f"  {tokens.split()[0]}  (AMT_0 = $0-10)")

print("*" * 50)

print("\nOnline transaction (no physical location):")
tokens = quick_tokenize({"Use Chip": "Online Transaction", "Zip": "00000", "Merchant State": ""})
print(f"  Tokens: {tokens}")

print("*" * 50)

print("\nThe tokenizer maps all amounts to 7 log-scale bins, preserving magnitude")
print("while avoiding the sparsity of exact dollar amounts.")

Large amount ($8,500):
AMT_6 MERCH_898 CAT_RETAIL MCC_5411 HOUR_09 DOW_2 MONTH_01 CARD_0 CHIP_CHIP ZIP3_951 STATE_CA CUST_1001
  AMT_6  (AMT_6 = $5,000+)
**************************************************

Small amount ($0.50):
AMT_0 MERCH_898 CAT_RETAIL MCC_5411 HOUR_09 DOW_2 MONTH_01 CARD_0 CHIP_CHIP ZIP3_951 STATE_CA CUST_1001
  AMT_0  (AMT_0 = $0-10)
**************************************************

Online transaction (no physical location):
  Tokens: AMT_1 MERCH_898 CAT_RETAIL MCC_5411 HOUR_09 DOW_2 MONTH_01 CARD_0 CHIP_ONLINE ZIP3_000 STATE_XX CUST_1001
**************************************************

The tokenizer maps all amounts to 7 log-scale bins, preserving magnitude
while avoiding the sparsity of exact dollar amounts.


## 2. Generate Tokenized Corpora (Train / Val / Test)

Convert each **temporal** split into text sequences for causal language modeling. Transactions are **grouped by (user, card)**, chunked into sequences of **~315 transactions**, and formatted as:

```
<bos> AMT_x MERCH_x ... CUST_x <sep> AMT_y MERCH_y ... CUST_y <sep> AMT_z MERCH_z ... CUST_z <sep> ... <eos>
```

Each sequence is ~4096 tokens, matching the foundation model training context window.

In [11]:
import time
import cudf

import rmm
# Enable managed memory to prevent OOM by spilling to system RAM
rmm.reinitialize(managed_memory=True)

MERCHANT_HASH_SIZE = 2000
CHUNK_SIZE = 315  # ~4096 tokens with 12 tok/txn + separators

# CORPUS_DIR.mkdir(parents=True, exist_ok=True)

splits = [
    ("train", TRAIN_DATA, CORPUS_PATH),
    ("val",   VAL_DATA,   VAL_CORPUS_PATH),
    ("test",  TEST_DATA,  TEST_CORPUS_PATH),
]

for split_name, parquet_path, corpus_path in splits:
    # if corpus_path.exists():
    #     n_lines = sum(1 for _ in open(corpus_path))
    #     print(f"[{split_name}] Corpus already exists: {n_lines:,} sequences")
    #     continue

    print(f"\n{'='*60}")
    print(f"Generating decoder corpus for {split_name}")
    print(f"{'='*60}")

    t0 = time.time()
    gdf = cudf.read_parquet(str(parquet_path))
    print(f"  Loaded {split_name}: {len(gdf):,} rows in {time.time()-t0:.1f}s")

    pip = FinancialTokenizerPipeline(merchant_hash_size=MERCHANT_HASH_SIZE)
    gdf_proc = pip.preprocess(gdf)
    pip.fit(gdf_proc)
    token_df = pip.transform(gdf_proc)

    # Identify grouping columns for chunking by (user, card)
    group_cols = []
    for col_name in ["user", "User", "cust"]:
        if col_name in gdf_proc.columns:
            group_cols.append(col_name)
            break
    for col_name in ["card", "Card", "card_id"]:
        if col_name in gdf_proc.columns:
            group_cols.append(col_name)
            break

    if not group_cols:
        group_cols = [gdf_proc.columns[0]]

    print(f"  Grouping by: {group_cols}")
    print(f"  Chunk size: {CHUNK_SIZE} transactions (~{CHUNK_SIZE * 13} tokens)")

    corpus_lines = pip.to_corpus_lines(
        token_df, gdf_proc, group_cols, chunk_size=CHUNK_SIZE
    )

    with open(corpus_path, "w") as f:
        for line in corpus_lines:
            f.write(line + "\n")

    elapsed = time.time() - t0
    print(f"  Generated {len(corpus_lines):,} sequences in {elapsed:.1f}s")
    print(f"  Saved to: {corpus_path}")

    sample = corpus_lines[0]
    n_tokens = len(sample.split())
    n_seps = sample.count("<sep>")
    print(f"  Sample: {n_tokens} tokens, {n_seps+1} transactions")
    print(f"  Preview: {sample[:200]}...")


Generating decoder corpus for train
  Loaded train: 19,508,123 rows in 11.4s
  Grouping by: ['user', 'card']
  Chunk size: 315 transactions (~4095 tokens)
  Generated 64,335 sequences in 110.5s
  Saved to: /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/train_corpus.txt
  Sample: 4096 tokens, 315 transactions
  Preview: <bos> AMT_3 MERCH_898 CAT_RETAIL MCC_5300 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_1 MERCH_1100 CAT_RETAIL MCC_5411 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917...

Generating decoder corpus for val
  Loaded val: 2,435,982 rows in 5.3s
  Grouping by: ['user', 'card']
  Chunk size: 315 transactions (~4095 tokens)
  Generated 9,739 sequences in 33.0s
  Saved to: /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/val_corpus.txt
  Sample: 4096 tokens, 315 transactions
  Preview: <bos> AMT_3 MERCH_728 CAT_RETAIL MCC_5300 HOUR_06 DOW_0 MONTH_05 CARD_0 CHIP_CHIP ZIP3_917 STATE_CA CU

## 3. Verify Output

In [12]:
import os

print("Corpus Summary:")
print("=" * 60)
for name, path in [("Train", CORPUS_PATH), ("Val", VAL_CORPUS_PATH), ("Test", TEST_CORPUS_PATH)]:
    n_lines = sum(1 for _ in open(path))
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {name:6s}: {n_lines:>8,} sequences  ({size_mb:>7.1f} MB)  {path}")

print(f"\nFirst 3 lines of {CORPUS_PATH}:")
with open(CORPUS_PATH, 'r') as f:
    for _ in range(3):
        line = f.readline().strip()
        print(f"  {line[:120]}...")

Corpus Summary:
  Train :   64,335 sequences  ( 2110.3 MB)  /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/train_corpus.txt
  Val   :    9,739 sequences  (  262.4 MB)  /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/val_corpus.txt
  Test  :   10,651 sequences  (  263.2 MB)  /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/test_corpus.txt

First 3 lines of /content/drive/MyDrive/Colab-Notebooks/Foundation-Transaction/decoder_corpus/train_corpus.txt:
  <bos> AMT_3 MERCH_898 CAT_RETAIL MCC_5300 HOUR_06 DOW_6 MONTH_09 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_1 ...
  <bos> AMT_1 MERCH_1100 CAT_RETAIL MCC_5411 HOUR_06 DOW_3 MONTH_12 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> AMT_2...
  <bos> AMT_1 MERCH_1259 CAT_MISC_STORES MCC_5912 HOUR_19 DOW_4 MONTH_08 CARD_0 CHIP_SWIPE ZIP3_917 STATE_CA CUST_0 <sep> ...


## Summary

The custom Financial Tokenizer produces 3-4x fewer tokens per transaction than GPT-2 BPE (12 tokens vs 30-50+), enabling ~315 transactions per 4096-token sequence. Each token captures a complete financial concept rather than arbitrary subwords.

### Outputs:

* **training corpus**: data/decoder_corpus/train_corpus.txt (~ 64K sequences)
* **validation corpus**: data/decoder_corpus/val_corpus.txt (~ 10K sequences)
* **test corpus**: data/decoder_corpus/test_corpus.txt (~ 11K sequences)